# STAR-RIS RSMA - main experiment (V4 shard workflow)

Quy trinh phan tan (KHONG chay 40 run trong mot process):

1. Moi Kaggle job chay DUNG MOT cap algorithm-seed bang `--train-only`
   (cell 3 sinh danh sach 40 lenh; cell 4 chay MOT cap do ban chon).
2. Thu du 40 shard (5 thuat toan x 8 seed) ve mot thu muc
   `results_revised/shards/<GROUP_ID>/`.
3. Job cuoi cung chay `--aggregate-only --final-paper` (cell 5): xac minh
   source/config/checkpoint SHA cua tung shard, nap history da hash de ve
   hinh hoi tu, va TU CHOI neu thieu/du/trung cap algorithm-seed.

Quy tac: tune chi bang validation; khong dong den locked test truoc aggregate
cuoi; moi lan tune ghi vao tuning_history.csv. Chi tiet:
`KAGGLE_RERUN_CHECKLIST.md`.

In [ ]:
# Setup: repo + dependency pinned + test suite phai PASS truoc khi train dai.
import os, sys, subprocess
REPO = "/kaggle/working/STAR_RIS_RSMA_MADDPG"  # chinh lai neu chay local
if not os.path.isdir(REPO):
    REPO = os.path.abspath("..")
os.chdir(REPO)
print("Repo root:", REPO)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=False)
r = subprocess.run([sys.executable, "-m", "pytest", "tests", "-q"], capture_output=True, text=True)
print(r.stdout[-2000:])
assert r.returncode == 0, "Test suite failed - khong chay thi nghiem voi code loi"

In [ ]:
# Cau hinh nhom shard. GROUP_ID co dinh de moi job ghi vao cung mot nhom.
from main import load_config, source_tree_sha256

GROUP_ID = "paper_v1"          # dat mot lan, dung chung cho ca 40 job
CONFIG = "config/config.yaml"
cfg = load_config(CONFIG)       # resolve + verify seed_split SHA-256
ALGOS = cfg["evaluation"]["required_algorithms"]
SEEDS = cfg["evaluation"]["required_training_seeds"]
print("Source tree sha256:", source_tree_sha256()[:16])
print(f"Matrix: {len(ALGOS)} algorithms x {len(SEEDS)} seeds = {len(ALGOS)*len(SEEDS)} shard jobs")

In [ ]:
# Sinh danh sach 40 lenh train-only (moi lenh = mot Kaggle job rieng).
commands = [
    f"python main.py --config {CONFIG} --train-only "
    f"--algos {algo} --seeds {seed} --run-id {GROUP_ID}"
    for algo in ALGOS for seed in SEEDS
]
for c in commands:
    print(c)
with open("shard_jobs.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(commands) + "\n")
print(f"\nDa ghi {len(commands)} lenh vao shard_jobs.txt")

In [ ]:
# Chay DUNG MOT cap algorithm-seed trong job nay (chon qua bien duoi day
# hoac bien moi truong SHARD_ALGO / SHARD_SEED cua Kaggle job).
import subprocess, sys, os
SHARD_ALGO = os.environ.get("SHARD_ALGO", ALGOS[0])
SHARD_SEED = int(os.environ.get("SHARD_SEED", SEEDS[0]))
cmd = [sys.executable, "main.py", "--config", CONFIG, "--train-only",
       "--algos", SHARD_ALGO, "--seeds", str(SHARD_SEED),
       "--run-id", GROUP_ID]
print(" ".join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
# AGGREGATE (job cuoi, sau khi thu du 40 shard):
# --final-paper bat kiem tra nghiem ngat ma tran algorithm x seed da dang ky
# + dong nhat source_sha/config_sha; history duoc xac minh hash va dung de
# ve cac hinh hoi tu. Voi pilot thieu shard, thay --final-paper bang
# --allow-partial (khong dung cho bang so lieu Chuong 4).
import subprocess, sys
cmd = [sys.executable, "main.py", "--aggregate-only",
       "--load-shards", f"results_revised/shards/{GROUP_ID}",
       "--final-paper", "--run-id", f"{GROUP_ID}_final"]
print(" ".join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
# Kiem tra dau ra aggregate: bang, hinh hoi tu, provenance.
import os, glob, pandas as pd
out = f"results_revised/{GROUP_ID}_final"
display(pd.read_csv(os.path.join(out, "completed_runs.csv")))
display(pd.read_csv(os.path.join(out, "tables", "algorithm_comparison.csv")))
display(pd.read_csv(os.path.join(out, "tables", "significance.csv")))
expected_figs = ["training_convergence", "training_sum_rate",
                 "training_user_qos_fraction", "qos_lambda", "reward_decomposition"]
for name in expected_figs:
    hits = glob.glob(os.path.join(out, "figures", name + ".*"))
    print(("OK   " if hits else "MISS ") + name)
raw = pd.read_csv(os.path.join(out, "tables", "results_raw.csv"))
assert {"source_sha", "config_sha", "checkpoint_sha", "scenario_id"} <= set(raw.columns)
print("results_raw.csv rows:", len(raw))